# 短期记忆

## 1.基于内存的持久化器

### 1.1 举例1:没有记忆

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL
)

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model = model,
    tools = []
)

print("\n第一轮对话:")
res1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]
})
print(f"Agent: {res1['messages'][-1].content}")

print("\n第二轮对话:")
res2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]
})
print(f"Agent: {res2['messages'][-1].content}")

### 1.2 举例2：拥有记忆

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

checkpoint = InMemorySaver()   # 1、创建了内存级别的记忆存储

agent = create_agent(
    model = model,
    tools = [],
    checkpointer=checkpoint   # 2、让Agent具备了存储的能力
)

# 3、同一个thread_id共享记忆的
config = {
    "configurable" : {
        "thread_id": "1"
    }
}

# 同一个thread_id共享记忆的
print("\n第一轮对话:")
res1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]},
    config = config
)
print(f"Agent: {res1['messages'][-1].content}")

print("\n第二轮对话:")
res2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
config = config)
print(f"Agent: {res2['messages'][-1].content}")

In [ ]:
from rich import print as rprint

thread_1_state = agent.get_state(config)
rprint(thread_1_state)

继续：

In [ ]:
print("\n第三轮对话:")
res3 = agent.invoke({
    "messages": [HumanMessage("我刚才问了啥问题？")]},
config = config)
print(f"Agent: {res3['messages'][-1].content}")

继续：不同的thread_id不共享记忆

In [15]:
config1 = {
    "configurable": {
        "thread_id": "2"
    }
}

print("\n第四轮对话:")
res4 = agent.invoke({
    "messages": [HumanMessage("我叫啥名字？")]},
config = config1)
print(f"Agent: {res4['messages'][-1].content}")


第四轮对话:
Agent: 我真的不知道呀，因为你还没告诉我你的名字。😊  
你想让我怎么称呼你？告诉我，我就记住啦。


In [ ]:
thread_2_state = agent.get_state(config1)

rprint(thread_2_state)